# Phase 6.4B.1/C — Primary + Supplementary Evidence and Specification Extraction
Six frozen development cases. Preserve all primary evidence, add at most four chunks / 4,000 evidence characters per case. No new retrieval index, library context, or historical claims in requests. First run the offline merge and inspect it. Maximum six new DeepSeek attempts, no retry, one Colab session; persistent cache must not be deleted. Claims are candidates, not confirmed clinical requirements.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib, shutil, json
REPO_URL='https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH='simplified-rag-v3'
ROOT=Path('/content/tfl-simplified-rag-v3')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[retrieval,test]','openai'],check=True)
SRC=ROOT/'src'
sys.path.insert(0,str(SRC))
for name in list(sys.modules):
    if name=='tfl_agent' or name.startswith('tfl_agent.'):
        del sys.modules[name]
for p in SRC.joinpath('tfl_agent').rglob('__pycache__'):
    shutil.rmtree(p)
importlib.invalidate_caches()
import tfl_agent
assert Path(tfl_agent.__file__).resolve().is_relative_to(SRC.resolve())
print('Commit:',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())
subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests/test_phase6_specification_pilot.py')],check=True)

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
from tfl_agent.agent.phase6_specification_pilot import (
    ARCHIVE_NAME,ARCHIVE_SHA,sha,prepare,run_pilot,export,BUDGET,MODEL)
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/Colab Notebooks/agent/midterm demo')
SOURCE=BASE/'Phase64B_Field_Retrieval_Outputs'
ARCHIVE=SOURCE/ARCHIVE_NAME
if not ARCHIVE.is_file() or sha(ARCHIVE.read_bytes())!=ARCHIVE_SHA:
    matches=[p for p in SOURCE.glob('*.zip') if sha(p.read_bytes())==ARCHIVE_SHA]
    if len(matches)!=1:
        raise FileNotFoundError(f'Expected {ARCHIVE} with SHA256 {ARCHIVE_SHA}; matches: {matches}')
    ARCHIVE=matches[0]
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
DEST=BASE/'Phase64C_Specification_Pilot_Outputs'
DEST.mkdir(parents=True,exist_ok=True)
CACHE=DEST/'Frozen_Specification_Cache_v1'
OUT=Path('/content/Phase64C_Specification_Pilot')/RUN_ID
requests,merge=prepare(ARCHIVE,OUT/'Merge')
assert merge['primary_chunks']==53 and merge['historical_quote_hits']>=60
assert len(requests)==6 and merge['new_llm_calls']==0
assert all(set(r['payload'])=={'target','sap_evidence','allowed_fields'} for r in requests)
print(json.dumps(merge,indent=2))
import pandas as pd
from IPython.display import display
display(pd.read_csv(OUT/'Merge'/'Merge_Budget.csv'))
print('Model:',MODEL,'| New attempt cap:',BUDGET)
print('Already reserved:',len(list(CACHE.glob('*.attempt.json'))))
print('Verified archive:',ARCHIVE)

## Six-attempt extraction
The development conversation authorized up to six additional DeepSeek attempts. Run this cell only if authorized to send these SAP excerpts to DeepSeek. Use DEEPSEEK_API_KEY in Colab Secrets or enter the key privately. No retry, including transport or validation failures. Do not delete the persistent cache or run concurrent sessions. If interrupted, rerun this cell; previously reserved requests are skipped.

In [ ]:
from google.colab import userdata
from getpass import getpass
AUTHORIZE_SIX_ATTEMPTS=True #@param {type:"boolean"}
if not AUTHORIZE_SIX_ATTEMPTS:
    raise RuntimeError('Authorization is required for new requests.')
try:
    key=userdata.get('DEEPSEEK_API_KEY')
except Exception:
    key=None
if not key:
    key=getpass('DeepSeek API key (hidden): ')
if not key:
    raise ValueError('Missing API key')
try:
    run_pilot(requests,CACHE,key,authorized=AUTHORIZE_SIX_ATTEMPTS)
finally:
    del key

In [ ]:
report,specs=export(requests,CACHE,OUT/'Extraction')
assert len(specs)==6 and report['new_attempts_reserved']<=6 and not report['generation_authorized']
print(json.dumps(report,indent=2))
coverage=pd.read_csv(OUT/'Extraction'/'Field_Coverage.csv')
display(coverage.pivot(index='request_id',columns='field',values='status'))
saved=Path(shutil.make_archive(str(DEST/RUN_ID),'zip',root_dir=OUT))
print('Saved:',saved)
print('Files:',sorted(str(p.relative_to(OUT)) for p in OUT.rglob('*') if p.is_file()))
print('Share only if authorized to disclose SAP excerpts and responses.')

## Interpretation
Merge is zero-call: supplement priority uses lexical target anchors, general-rule clues, and stored retrieval ranks, not independent applicability. All baseline chunks are preserved; historical quote overlap is diagnostic only. A parsed response may contain rejected claims. Exact spans refer to normalized SAP blocks. Title evidence and SAP evidence remain separate. Candidate coverage is not correctness; unknown fields are not absent requirements. No independent semantic validation, cross-field conflict resolution, template alignment, or DOCX generation. These are next-stage tasks.